## Lab 11

If both CFs are positive (supporting evidence): CFcombined = CF1 + CF2 * (1 - CF1)

In [1]:
import clips 
import logging

# Setup working environment
logging.basicConfig(level=logging.INFO,format='%(message)s')
    
env = clips.Environment()
router = clips.LoggingRouter()
env.add_router(router)

env.build("""
(deftemplate symptom
   (slot name)
   (slot cf))   ; certainty factor """)

env.build("""
(defrule flu-rule
   (symptom (name fever) (cf ?cf1))
   (symptom (name cough) (cf ?cf2))
   =>
   (bind ?cf (+ ?cf1 (* ?cf2 (- 1 ?cf1)))) ; CFcombined = CF1 + CF2 * (1 - CF1)
   (printout t "Flu inferred with CF = " ?cf crlf)) """)

env.eval("(assert (symptom (name fever) (cf 0.8)))")
env.eval("(assert (symptom (name cough) (cf 0.6)))") 

env.run()


Flu inferred with CF = 0.92


1

If both CFs are negative (supporting evidence): CFcombined = CF1 + CF2 * (1 + CF1)

In [2]:
import clips 
import logging

# Setup working environment
logging.basicConfig(level=logging.INFO,format='%(message)s')
    
env = clips.Environment()
router = clips.LoggingRouter()
env.add_router(router)

env.build("""
(deftemplate symptom
   (slot name)
   (slot cf))   ; certainty factor """)

env.build("""
(defrule flu-rule
   (symptom (name fever) (cf ?cf1))
   (symptom (name cough) (cf ?cf2))
   =>
   (bind ?cf (+ ?cf1 (* ?cf2 (+ 1 ?cf1)))) ; CFcombined = CF1 + CF2 * (1 + CF1)
   (printout t "Flu inferred with CF = " ?cf crlf)) """)

env.eval("(assert (symptom (name fever) (cf -0.8)))")
env.eval("(assert (symptom (name cough) (cf -0.6)))") 

env.run()


Flu inferred with CF = -0.92


1

If both CFs are negative (supporting evidence): CFcombined = (CF1 + CF2) / (1 - min(|CF1|,|CF2|))

In [3]:
import clips 
import logging

# Setup working environment
logging.basicConfig(level=logging.INFO,format='%(message)s')
    
env = clips.Environment()
router = clips.LoggingRouter()
env.add_router(router)

env.build("""
(deftemplate symptom
   (slot name)
   (slot cf))   ; certainty factor """)

env.build("""
(defrule flu-rule
   (symptom (name fever) (cf ?cf1))
   (symptom (name cough) (cf ?cf2))
   =>
   (bind ?cf
      (if (and (>= ?cf1 0) (>= ?cf2 0)) then
          (+ ?cf1 (* ?cf2 (- 1 ?cf1))) ; supporting evidence
      else
          (if (and (<= ?cf1 0) (<= ?cf2 0)) then
              (+ ?cf1 (* ?cf2 (+ 1 ?cf1))) ; negative evidence
          else
              (/ (+ ?cf1 ?cf2) (- 1 (min (abs ?cf1) (abs ?cf2)))))))
   (printout t "Flu inferred with CF = " ?cf crlf))
""")

env.eval("(assert (symptom (name fever) (cf 0.8)))")
env.eval("(assert (symptom (name cough) (cf -0.6)))") 

env.run()


Flu inferred with CF = 0.5


1